# Imports and preprocessing of train data

In [11]:
import csv
import numpy as np
from pathlib import Path
import sys, os
import json
import importlib
sys.path.append(os.path.join(os.getcwd(), '..'))

from utils import eda_helpers, all_functions, cv, sampling_methods
import implementations

importlib.reload(eda_helpers)
importlib.reload(all_functions)
importlib.reload(cv)
importlib.reload(sampling_methods)
importlib.reload(implementations)

from utils.eda_helpers import clean_data, read_csv_numeric
from utils.all_functions import sigmoid
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    weighted_logistic_regression,
)

file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')

In [2]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))

In [3]:
final_feature_set = ['_CASTHM1', 'FLUSHOT6', 'METVL11_', '_PASTRNG', 'EDUCA', 'CHCCOPD1', 'DRADVISE', 
                     'CHCKIDNY', 'ARTHSOCL', 'MENTHLTH', 'SMOKE100', 'DOCTDIAB', 'DIABAGE2', 'VETERAN3', 
                     'CHKHEMO3', 'SEX', '_DRNKWEK', 'JOINPAIN', 'MARITAL', 'EXERANY2', '_PACAT1', 'INSULIN', 
                     'DIFFALON', 'FRUIT1', 'QLACTLM2', 'BLIND', 'PADUR2_', 'PAFREQ2_', 'LMTJOIN3', 'CHOLCHK', 
                     'POORHLTH', 'DIABETE3', '_FRUTSUM', 'PERSDOC2', 'EXRACT21', 'FEETCHK', '_VEGESUM', 'PNEUVAC3', 
                     'BPHIGH4', 'GENHLTH', 'EXRACT11', '_BMI5', 'DECIDE', '_ASTHMS1', '_MRACE1', '_LMTSCL1', 'ORNGDAY_', 
                     'MSCODE', 'PHYSHLTH', 'TOLDHI2', 'PADUR1_', 'DIFFWALK', 'BEANDAY_', 'CHCSCNCR', 'PAFREQ1_', '_AIDTST3', 
                     'FLSHTMY2', 'PA1VIGM_', 'METVL21_', 'SXORIENT', 'ARTHDIS2', 'PA1MIN_', 'MAXVO2_', '_RACE_G1', 'LONGWTCH', 
                     'SCNTWRK1', 'FEETCHK2', 'EMPLOY1', 'BLOODCHO', 'USENOW3', 'DRNKANY5', '_PAREC1', 'DIFFDRES', 'USEEQUIP', 
                     'ADDEPEV2', '_AGE80', 'HTM4', 'INCOME2', 'CVDSTRK3', 'CHCOCNCR', 'MEDCOST', 'PREDIAB1', 'BPMEDS', 'HAVARTH3', 
                     'CHECKUP1', 'TRNSGNDR', 'CHILDREN']

In [4]:
del x_indices["Id"]
x_data = clean_data(x_data, x_indices)

In [29]:
# --- Keep only the columns of interest ---
x_data_interested = x_data[:, [x_indices[col] for col in final_feature_set]]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Drop columns that are all NaN ---
valid_cols = ~np.isnan(x_data_interested).all(axis=0)
x_data_interested = x_data_interested[:, valid_cols]
interesting_columns = [col for col, keep in zip(final_feature_set, valid_cols) if keep]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Impute remaining NaNs with column mean ---
col_mean = np.nanmean(x_data_interested, axis=0)
inds = np.where(np.isnan(x_data_interested))
x_data_interested[inds] = np.take(col_mean, inds[1])

# --- Standardize once ---
train_means = np.mean(x_data_interested, axis=0)
train_stds = np.std(x_data_interested, axis=0)
x_data_standardized = (x_data_interested - train_means) / (train_stds + 1e-8)

# --- Prepare y ---
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.flatten()

# --- Checks ---
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data_standardized)))
print("Min std across features:", np.min(np.std(x_data_standardized, axis=0)))
print("Label distribution (mean):", np.mean(y))

Unique labels: [0. 1.]
Any NaN in features? False
Min std across features: 0.9999999002870875
Label distribution (mean): 0.08830207079403295


# Train weighted log reg

In [12]:
from utils.cv import train_logreg, predict
from utils.helpers import load_csv_data, create_csv_submission

In [7]:
# load test data
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')
x_indices_test, x_test = read_csv_numeric(os.path.join(DATA_FOLDER, "x_test.csv"))

# clean it
if "Id" in x_indices_test:
    del x_indices_test["Id"]
x_test = clean_data(x_test, x_indices_test)

In [36]:
# Keep only same features as in training 
x_test_interested = x_test[:, [x_indices_test[col] for col in interesting_columns]]

# Impute missing values using training column means 
inds_test = np.where(np.isnan(x_test_interested))
x_test_interested[inds_test] = np.take(col_mean, inds_test[1])

# Standardize using training mean/std 
train_means = np.mean(x_data_interested, axis=0)
train_stds = np.std(x_data_interested, axis=0)
x_test_standardized = (x_test_interested - train_means) / (train_stds + 1e-8)

In [37]:
print(x_data_standardized.shape)
print(x_test_standardized.shape)


(328135, 87)
(109379, 87)


In [ ]:
# check if all good
# z = x_test_standardized @ w
# print("Mean of (x_test @ w):", np.mean(z))
# print("Min/Max of (x_test @ w):", np.min(z), np.max(z))
# print("Sigmoid range:", sigmoid(z).min(), sigmoid(z).max())


Mean of (x_test @ w): -0.002347254361829618
Min/Max of (x_test @ w): -2.5069803191280293 5.3426358172590875
Sigmoid range: 0.07537028018709774 0.9952395242116676


In [98]:
lambda_ = 0.01
gamma = 0.4
max_iters = 500
threshold = 0.80

print("\n=== Training Weighted Logistic Regression ===")
w, final_loss = train_logreg(
    y=y,
    tx=x_data_standardized,
    kind="w_logistic",
    gamma=gamma,
    lambda_=lambda_,
    max_iters=max_iters,
    tol=None,
    verbose=True
)
print(f"Final training loss: {final_loss:.6f}")


=== Training Weighted Logistic Regression ===


[w_logistic] Training finished — final loss: 0.519229
Final training loss: 0.519229


In [99]:
# predict on test set
print("\n=== Generating Predictions ===")
y_pred_test = predict(x_test_standardized, w, threshold=threshold)

# Convert 0,1 to -1,1 for AI Crowd submission
y_pred_test = np.where(y_pred_test == 1, 1, -1)


=== Generating Predictions ===


In [100]:
# --- Load test IDs and create submission file ---
test_ids = np.genfromtxt(
    os.path.join(DATA_FOLDER, "x_test.csv"),
    delimiter=",",
    skip_header=1,
    usecols=0,
    dtype=int
)

submission_path = os.path.join(file_path, f"submission_weighed_log_reg_gamma{gamma}_lambda{lambda_}_thr{threshold}.csv")
create_csv_submission(test_ids, y_pred_test, submission_path)
print(f"\nSubmission file successfully created: {submission_path}")


Submission file successfully created: c:\Users\emmaa\OneDrive\Documents\0. EPFL\courses\CS_433 - Machine Learning\cs433_medical_ml\notebooks\submission_weighed_log_reg_gamma0.4_lambda0.01_thr0.8.csv


In [101]:
# --- Sanity checks ---
print("Unique predictions:", np.unique(y_pred_test))
print("Number of test samples:", len(y_pred_test))
print("First 10 predictions:")
print(np.column_stack((test_ids[:10], y_pred_test[:10])))

n_pos = np.sum(y_pred_test == 1)
n_neg = np.sum(y_pred_test == -1)

print(f"Number of +1 predictions: {n_pos}")
print(f"Number of -1 predictions: {n_neg}")
print(f"Total: {len(y_pred_test)}")

Unique predictions: [-1  1]
Number of test samples: 109379
First 10 predictions:
[[328135     -1]
 [328136     -1]
 [328137     -1]
 [328138     -1]
 [328139     -1]
 [328140     -1]
 [328141     -1]
 [328142     -1]
 [328143      1]
 [328144     -1]]
Number of +1 predictions: 13788
Number of -1 predictions: 95591
Total: 109379


In [ ]:
from collections import defaultdict
def euclidean_distance(x,y) : 
    n = len(x)
    return np.sqrt(np.sum([(x[i] - y[i])**2 for i in range(n)]))

def k_means(X : np.array, k) : 
    rows, cols = len(X), len(X[0])
    X_shuffled = np.random.shuffle(X)
    centroids = X_shuffled[:k]
    centroids_old = np.arange(k*cols).reshape(k, cols)
    nearest_neighbours = defaultdict(list)

    while centroids != centroids_old : 

        distances = [[euclidean_distance(x, centroid) for centroid in centroids] for x in X]
        # assign NNs

        for j in range(len(distances)) : 
            i = np.argmin(distances[j])
            nearest_neighbours[i].append(X[j])

        # recompute centroids
        centroids_old =  centroids
        centroids = [np.mean(nearest_neighbours[k]) for k in range(rows)]

    return nearest_neighbours
